# Train an ASL Letter + Number Recognizer (A–Z, 0–9)

Trains a **small `scikit-learn` classifier** on **MediaPipe hand landmarks** for the sign-language learning app. MediaPipe is reused as-is — only a tiny classifier head is trained.

**Data sources (Kaggle):**
- Numbers 0–9: `rayeed045/american-sign-language-digit-dataset` — ships a **landmark CSV per digit**, loaded directly (Section 4A).
- Letters A–Z: `grassknoted/asl-alphabet` — raw hand photos; landmarks are extracted with MediaPipe (Section 4B).

**Output:** `asl_landmark_model.joblib` + `labels.json` for the backend.
Runtime: CPU is fine. J/Z are motion letters (handle specially in the app); some letter/number handshapes overlap (2≈V, 6≈W, 9≈F, 0≈O) — disambiguate by mode.


## 1. Install dependencies

Colab ships **NumPy 2**, but official MediaPipe pins numpy<2, and *downgrading* numpy on Colab corrupts the env. So we **keep numpy 2** and install **`mediapipe-numpy2`** (numpy-2 build; still `import mediapipe as mp`). No downgrade, no forced restart.

> If you earlier ran a numpy<2 version and hit `numpy._core.multiarray failed to import`: **Runtime → Disconnect and delete runtime**, then run this. `mediapipe-numpy2` is an unofficial patched wheel ([source](https://github.com/cansik/mediapipe-numpy2)).


In [ ]:
!pip -q uninstall -y mediapipe >/dev/null 2>&1
!pip -q install mediapipe-numpy2 scikit-learn pandas joblib matplotlib tqdm kaggle
print("Installed. Continue to Section 1b (no restart needed).")

## 1b. Verify the environment

In [ ]:
import numpy, cv2, mediapipe as mp
print("numpy", numpy.__version__, "| opencv", cv2.__version__, "| mediapipe", mp.__version__)
_ = mp.solutions.hands          # must not raise
print("Environment OK - continue to Section 2")
# If this errors with 'numpy._core.multiarray failed to import':
#   Runtime -> Disconnect and delete runtime, then re-run Section 1.

## 2. Download the datasets from Kaggle

Get a token at **kaggle.com → Settings → Create New Token** (`kaggle.json`), then run and upload it. *(Or upload/unzip into `data/letters` and `data/digits` yourself.)*


In [ ]:
from google.colab import files
import os
print("Upload your kaggle.json:")
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Numbers: real photos + a landmark CSV per digit.
!kaggle datasets download -d rayeed045/american-sign-language-digit-dataset -p data/digits --unzip

# Letters: RAW photos. (The 'srisahithis' A-Z set only has landmark-annotated images -- red dots drawn
# ON the hand -- which MediaPipe cannot re-detect and which occlude the hand, so it is unusable here.)
!kaggle datasets download -d grassknoted/asl-alphabet -p data/letters --unzip
print("Downloaded. Inspect in Section 3.")

## 3. Config, helpers, and inspection

Defines the landmark normalization used by **both** training and runtime, plus label parsing, and prints the folder tree / CSVs.


In [ ]:
import os, glob, re, cv2, numpy as np, pandas as pd
from tqdm.auto import tqdm
import mediapipe as mp
mp_hands = mp.solutions.hands

FEATURES_CSV = "features.csv"
FEAT_COLS = [f"{a}{i}" for i in range(21) for a in ("x", "y", "z")]   # 63 output feature names

WORD_TO_DIGIT = {"zero":"0","one":"1","two":"2","three":"3","four":"4",
                 "five":"5","six":"6","seven":"7","eight":"8","nine":"9"}
SKIP = {"space","del","delete","nothing","blank","background","test","train"}
SKIP_PATH_SUBSTR = ["output images"]   # skip landmark-overlay copies in the digit dataset

def normalize_label(name):
    n = str(name).strip().lower()
    if n in SKIP: return None
    if n in WORD_TO_DIGIT: return WORD_TO_DIGIT[n]
    m = re.fullmatch(r"(?:sign|letter|digit|class|label)?[ _\-]*([a-z0-9])", n)
    return m.group(1).upper() if m else None

def label_from_path(path):
    for part in reversed(os.path.normpath(path).split(os.sep)[:-1]):
        lab = normalize_label(part)
        if lab is not None:
            return lab
    return None

def list_images(root):
    out = []
    for ext in (".jpg", ".jpeg", ".png", ".bmp"):
        out += glob.glob(os.path.join(root, "**", "*" + ext), recursive=True)
    return [p for p in out if not any(s in p.lower() for s in SKIP_PATH_SUBSTR)]

def normalize_landmarks(pts):
    # pts: (21,3) image-normalized. Wrist->origin + unit scale. Returns 63-vector (matches runtime).
    pts = np.asarray(pts, dtype=np.float32).copy()
    pts -= pts[0]
    scale = np.linalg.norm(pts, axis=1).max()
    if scale > 1e-6: pts /= scale
    return pts.flatten()

def pad_image(img, frac=0.6, white=True):
    h, w = img.shape[:2]
    bd = cv2.BORDER_CONSTANT if white else cv2.BORDER_REPLICATE
    return cv2.copyMakeBorder(img, int(h*frac), int(h*frac), int(w*frac), int(w*frac), bd, value=(255,255,255))

def extract_from_image(img_bgr, hands, pad=True):
    if pad: img_bgr = pad_image(img_bgr)
    res = hands.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if not res.multi_hand_landmarks: return None
    hand = res.multi_hand_landmarks[0]
    handed = res.multi_handedness[0].classification[0].label if res.multi_handedness else "Right"
    pts = np.array([[lm.x, lm.y, lm.z] for lm in hand.landmark], dtype=np.float32)
    if handed == "Left": pts[:, 0] = -pts[:, 0]
    return normalize_landmarks(pts)

# ---- inspect ----
for root in ["data/digits", "data/letters"]:
    print("==", root)
    for dp, dirs, files in os.walk(root):
        d = dp[len(root):].count(os.sep)
        if d > 3: dirs[:] = []; continue
        imgs = [f for f in files if f.lower().endswith((".jpg",".jpeg",".png"))]
        print("  "*d + os.path.basename(dp) + f"/ [{len(imgs)} imgs, {len(files)-len(imgs)} other]")
print("\nCSV files:", glob.glob("data/**/*.csv", recursive=True)[:3], "...")

## 4A. Numbers 0–9 — load the provided landmark CSVs (no detection needed)

The digit dataset ships one CSV per digit with columns `x00..z20` + `Label`. We load them and apply the **same normalization** used at runtime.


In [ ]:
digit_csvs = glob.glob("data/digits/**/*.csv", recursive=True)
print("digit CSVs found:", len(digit_csvs))
csv_coords = [f"{a}{i:02d}" for i in range(21) for a in ("x", "y", "z")]   # x00,y00,z00,...

rows = []
for c in digit_csvs:
    d = pd.read_csv(c)
    if "Label" not in d.columns or not all(col in d.columns for col in csv_coords):
        print("  skip (unexpected columns):", c); continue
    for _, r in d.iterrows():
        lab = normalize_label(r["Label"])
        if lab is None: continue
        pts = r[csv_coords].to_numpy(dtype=np.float32).reshape(21, 3)
        rows.append([lab, *normalize_landmarks(pts).tolist()])

df_digits = pd.DataFrame(rows, columns=["label"] + FEAT_COLS)
print("digits:", len(df_digits), "samples")
print(df_digits["label"].value_counts().sort_index().to_dict())

## 4B. Letters A–Z — extract landmarks from the raw photos (grassknoted)

Uses MediaPipe on the raw `grassknoted/asl-alphabet` photos, with white-border padding + low threshold. `MAX_PER_LABEL` caps per-letter samples so extraction is minutes, not ~40 min (there are 3000 raw photos per letter).

**Tip:** before the full run, sanity-check detection on a small sample by temporarily setting `MAX_PER_LABEL = 40`. Expect a healthy rate on raw photos. If it is still ~0, run the letters diagnostic (try bigger `pad`/upscale).


In [ ]:
LETTERS_ROOT  = "data/letters"   # grassknoted unzips under here (nested asl_alphabet_train/.../A/...)
MAX_PER_LABEL = 500              # detected samples to keep per letter (raise for more accuracy)

rows, processed, detected, kept = [], 0, 0, {}
with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.3) as hands:
    for path in tqdm(list_images(LETTERS_ROOT), desc="letters"):
        lab = label_from_path(path)
        if lab is None or lab.isdigit():           # letters only here
            continue
        if kept.get(lab, 0) >= MAX_PER_LABEL:      # enough for this letter -> skip (cheap)
            continue
        img = cv2.imread(path)
        if img is None: continue
        processed += 1
        feat = extract_from_image(img, hands)
        if feat is not None:
            detected += 1; kept[lab] = kept.get(lab, 0) + 1
            rows.append([lab, *feat.tolist()])

rate = detected / max(processed, 1)
print(f"letters detection rate: {detected}/{processed} = {rate:.1%}")
if rate < 0.3:
    print("LOW detection - run the letters diagnostic and increase pad/upscale, or check LETTERS_ROOT.")
df_letters = pd.DataFrame(rows, columns=["label"] + FEAT_COLS)
print("letters:", len(df_letters), "samples across", df_letters["label"].nunique(), "classes")
print(df_letters["label"].value_counts().sort_index().to_dict())

## 4C. Combine digits + letters into one training table

In [ ]:
parts = [p for p in [globals().get("df_digits"), globals().get("df_letters")] if p is not None and len(p)]
assert parts, "No data! Run 4A and/or 4B first."
df = pd.concat(parts, ignore_index=True)
df.to_csv(FEATURES_CSV, index=False)
print("combined:", len(df), "samples,", df["label"].nunique(), "classes")
print(df["label"].value_counts().sort_index().to_dict())

## 5. Train the small classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv(FEATURES_CSV)
X = df.drop(columns=["label"]).values
y = df["label"].astype(str).values
print("Training on", len(df), "samples,", df["label"].nunique(), "classes,", X.shape[1], "features")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

candidates = {
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale", probability=True)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}
scores = {}
for name, clf in candidates.items():
    clf.fit(X_tr, y_tr)
    scores[name] = clf.score(X_te, y_te)
    print(f"{name} test accuracy: {scores[name]:.3f}")

best = max(scores, key=scores.get)
model = candidates[best]
print(f"\nChosen model: {best}")
print(classification_report(y_te, model.predict(X_te)))

labels_sorted = sorted(np.unique(y))
cm = confusion_matrix(y_te, model.predict(X_te), labels=labels_sorted)
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(cm, display_labels=labels_sorted).plot(ax=ax, xticks_rotation="vertical", colorbar=False)
plt.title(f"Confusion matrix ({best})"); plt.show()

## 6. Save + download the model

In [ ]:
import joblib, json
joblib.dump(model, "asl_landmark_model.joblib")
json.dump(sorted(np.unique(y).tolist()), open("labels.json", "w"))
print("Saved asl_landmark_model.joblib and labels.json")
try:
    from google.colab import files
    files.download("asl_landmark_model.joblib")
    files.download("labels.json")
except Exception as e:
    print("Download manually from the Files panel.", e)

## 7. Quick inference test

Uses the same `extract_from_image` + `normalize_landmarks` the backend runs per frame.


In [ ]:
def predict_sign(image_path, model, top_k=3):
    img = cv2.imread(image_path)
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.3) as hands:
        feat = extract_from_image(img, hands)
    if feat is None:
        return "no hand detected", []
    proba = model.predict_proba([feat])[0]
    order = np.argsort(proba)[::-1][:top_k]
    return model.classes_[order[0]], [(model.classes_[i], float(proba[i])) for i in order]

# Example (replace with a real path):
# print(predict_sign("data/digits/American Sign Language Digits Dataset/5/Input Images - Sign 5/Sign 5 (1).jpeg", model))

## 8. Using this model in the app

```python
import joblib, json, numpy as np
model  = joblib.load("asl_landmark_model.joblib")
labels = json.load(open("labels.json"))
# per frame: feat = normalize_landmarks(<21x3 MediaPipe landmarks>); pred = model.predict([feat])[0]
```
- **Stability gate:** accept `pred` only after N consecutive frames above a probability threshold.
- **Words / multi-digit numbers:** advance a pointer over the target sequence (CAT→C,A,T; 25→2,5).
- **Letter vs digit mode:** restrict to digit classes when expecting a number (2≈V, 6≈W, 9≈F, 0≈O).
- **J / Z:** motion letters — don't grade from a single frame; use the animated-reference handling from the PRD.
